# Asistente de consulta de requerimientos y design rules (chatbot basado en reglas)


In [1]:
import sys, nltk
print("Python:", sys.version.split()[0])
print("NLTK:", nltk.__version__)


Python: 3.13.18
NLTK: 3.9.1


In [2]:
# Celda 1: librerías y carga de la base de conocimiento
import json, os, re, unicodedata
from nltk.stem.snowball import SnowballStemmer   # el stemmer no requiere descargar datos

KB_PATH = "knowledge_base.json"
if not os.path.exists(KB_PATH):
    from google.colab import files
    files.upload()                                # selecciona knowledge_base.json
with open(KB_PATH, encoding="utf-8") as f:
    KB = json.load(f)
RECORDS = KB["records"]
BY_ID = {r["id"]: r for r in RECORDS}
COLLECTIONS = KB["meta"]["collections"]
print(f"Base cargada: {len(RECORDS)} registros, {len(COLLECTIONS)} colecciones")


Saving knowledge_base.json to knowledge_base.json
Base cargada: 70 registros, 10 colecciones


In [3]:
# Celda 2: procesamiento de lenguaje (NLTK)
STEM_EN, STEM_ES = SnowballStemmer("english"), SnowballStemmer("spanish")

def norm(text):
    """Minúsculas y sin acentos."""
    text = unicodedata.normalize("NFKD", text.lower())
    return "".join(c for c in text if not unicodedata.combining(c))

STOP = set("""el la los las un una unos unas de del al en y o u que se por con para es son lo su sus mi me te
como cual cuales aplica aplican afecta afectan contra sobre hay hacer dame informacion info quiero necesito dime
the an of to in on and or is are for with what which how does do that this about my give show tell
apply applies affect affects against""".split())
GENERIC = set("regla reglas diseno design rule rules requerimiento requerimientos requisito requisitos "
              "requirement requirements rqt dr".split())

def words(text):
    return re.findall(r"[a-z0-9]+", norm(text))

def stems(word):
    # los tallos llevan etiqueta de idioma para que "hola" (ES) no coincida con "hole" (EN)
    return {("en", STEM_EN.stem(word)), ("es", STEM_ES.stem(word))}

# Índice: cada registro con los tallos de sus palabras clave (ES/EN) y de su título
for r in RECORDS:
    s = set()
    for phrase in r["keywords_en"] + [r["title"]]:
        for w in words(phrase):
            s.add(("en", STEM_EN.stem(w)))
    for phrase in r["keywords_es"]:
        for w in words(phrase):
            s.add(("es", STEM_ES.stem(w)))
    r["_stems"] = s

ID_RE = re.compile(r"\bdemo[-\s]?(dr|rqt)[-\s]?(\d{1,3})\b")
ES_HINT = {norm(w) for w in "hola que como quien cual cuales para regla aplica gracias adios ayuda temas dame informacion desviacion autor relacionados puedes hacer contra afecta pido otros estan".split()}
EN_HINT = set("hello hi the what which how who does apply applies rule rules thanks bye help topics give information deviation author related can you do against affect is are".split())

def has(t, *phrases):
    return any(re.search(r"\b" + re.escape(p) + r"\b", t) for p in phrases)

def detect_lang(t, current):
    w = set(words(t))
    es, en = len(w & ES_HINT), len(w & EN_HINT)
    return "es" if es > en else "en" if en > es else current


In [4]:
# Celda 3: mensajes del asistente (frases guía en ES/EN; contenido técnico en inglés)
NOTE = {"es": "Nota: datos sintéticos. Verifica en la fuente oficial.",
        "en": "Note: synthetic data. Verify in the official source."}
MSG = {
 "greet": {"es": "Hola. Soy el asistente de consulta de requerimientos y design rules de arneses. Los datos son sintéticos y de uso académico. Pregúntame por un tema (por ejemplo, clips) o por un ID.",
           "en": "Hello. I am the harness requirements and design rules assistant. The data is synthetic and for academic use. Ask me about a topic (for example, clips) or an ID."},
 "help": {"es": "Puedo buscar por tema (¿qué aplica para clips?), por tipo (design rules de grommets), por ID (DEMO-DR-007), mostrar registros relacionados (¿cuáles están relacionados?), el autor o la prioridad de un ID, y cómo pedir una desviación.",
          "en": "I can search by topic (what applies to clips?), by type (design rules for grommets), by ID (DEMO-DR-007), show related records, the author or priority of an ID, and how to request a deviation."},
 "topics": {"es": "Temas disponibles: ", "en": "Available topics: "},
 "thanks": {"es": "De nada. Si tienes otra consulta, escríbela.", "en": "You are welcome. If you have another question, just ask."},
 "bye": {"es": "Hasta pronto. Recuerda verificar los requisitos en la fuente oficial.", "en": "See you soon. Remember to verify requirements in the official source."},
 "fallback": {"es": "No entendí tu pregunta. Prueba con un tema (clips, grommets, ground), un ID (DEMO-DR-001) o escribe 'ayuda'.",
              "en": "I did not understand. Try a topic (clips, grommets, ground), an ID (DEMO-DR-001) or type 'help'."},
 "none": {"es": "No encontré registros sobre ese tema. Escribe 'temas' para ver los disponibles.",
          "en": "I found no records on that topic. Type 'topics' to see what is available."},
 "noid": {"es": "No existe ese ID en la base.", "en": "That ID is not in the knowledge base."},
 "found": {"es": "Encontré {n} registros. Los más relevantes:", "en": "I found {n} records. The most relevant:"},
 "more": {"es": "(+{n} más; escribe un ID para ver el detalle)", "en": "(+{n} more; type an ID for details)"},
 "also": {"es": "También relacionados:", "en": "Also related:"},
 "rel": {"es": "Relacionados con {id}:", "en": "Related to {id}:"},
 "norel": {"es": "Primero consulta un registro y luego pregunta por sus relacionados.", "en": "Look up a record first, then ask for its related records."},
}
def say(key, lang, **kw):
    return MSG[key][lang].format(**kw)

def ficha(r, lang):
    dev = "Allowed" if r["deviation"]["allowed"] else "Not allowed"
    return (f"ID: {r['id']} | Type: {r['type']} | Level: {r['requirement_level']} | Priority: {r['priority']}\n"
            f"Author: {r['author']}\nTitle: {r['title']}\nRule: {r['rule_text']}\n"
            f"Deviation: {dev}\nRelated: {', '.join(r['related_ids'])}\n{NOTE[lang]}")

def one_line(r):
    return f"  - {r['id']} ({r['priority']}): {r['title']}"


In [5]:
# Celda 4: clasificador de intenciones y generador de respuesta
ATTR_W = {"autor", "author", "prioridad", "priority", "obligatorio", "obligatoria", "mandatory", "nivel", "level"}
DEV_W = {"desviacion", "desviar", "deviation", "deviate", "waiver", "excepcion", "exception"}
REL_W = {"relacionados", "relacionado", "related", "otros", "similares", "similar"}
GREET_W = {"hola", "buenas", "buenos", "dias", "hello", "hi", "hey", "good", "morning"}
PRIO = {"Primary": 0, "Secondary": 1, "Tertiary": 2}
CUES = ("aplica", "aplican", "applies", "apply", "requerimiento", "requerimientos", "requirement", "requirements", "regla", "reglas", "rule", "rules", "rqt")

def new_state():
    return {"lang": "es", "last_id": None, "fails": 0}

def search(t):
    tipo = None
    if re.search(r"\bdesign rules?\b|\breglas? de diseno\b|\bdr\b", t):
        tipo = "Design Rule"
    elif re.search(r"\brqt\b|\brequirements?\b", t):
        tipo = "Requirement"
    q = [w for w in words(t) if w not in STOP and w not in GENERIC and len(w) > 1]
    scored = []
    for r in RECORDS:
        if tipo and r["type"] != tipo:
            continue
        score = sum(1 for w in q if stems(w) & r["_stems"])
        if score:
            # desempate: más coincidencias, RQT antes que DR, prioridad, ID
            scored.append((-score, r["precedence"], PRIO[r["priority"]], r["id"], r))
    scored.sort(key=lambda x: x[:4])
    return tipo, scored

def respond(text, st):
    t = norm(text)
    st["lang"] = L = detect_lang(t, st["lang"])
    m = ID_RE.search(t)
    rid = f"DEMO-{m.group(1).upper()}-{int(m.group(2)):03d}" if m else None
    w = set(words(ID_RE.sub(" ", t)))
    if rid and rid not in BY_ID:
        return "I6", say("noid", L)
    target = BY_ID.get(rid) or (BY_ID.get(st["last_id"]) if st["last_id"] else None)
    if target and w & DEV_W:                                    # I12 desviación
        st["last_id"] = target["id"]; d = target["deviation"]
        if d["allowed"]:
            return "I12", (f"{target['id']} - Deviation allowed: Yes\nMethod: {d['method']}\n"
                           f"Approver: {d['approver']}\nEvidence: {d['evidence']}")
        return "I12", f"{target['id']} - Deviation allowed: No\n{d['method']}"
    if target and w & ATTR_W:                                   # I11 atributos
        st["last_id"] = target["id"]
        return "I11", (f"{target['id']} - Author: {target['author']}. Priority: {target['priority']}. "
                       f"Level: {target['requirement_level']}. A requirement takes precedence over a design rule.")
    if rid:                                                     # I6 por identificador
        st["last_id"] = rid
        return "I6", ficha(BY_ID[rid], L)
    if w & REL_W:                                               # I7 relacionados
        if not target:
            return "I7", say("norel", L)
        lines = [one_line(BY_ID[i]) for i in target["related_ids"]]
        return "I7", say("rel", L, id=target["id"]) + "\n" + "\n".join(lines)
    if has(t, "ayuda", "help", "que puedes hacer", "what can you do"):
        return "I2", say("help", L)
    if has(t, "temas", "categorias", "colecciones", "topics", "categories", "collections"):
        return "I3", say("topics", L) + "; ".join(COLLECTIONS)
    if has(t, "adios", "bye", "goodbye", "hasta luego", "nos vemos", "chao"):
        return "I9", say("bye", L)
    if has(t, "gracias", "thanks", "thank you"):
        return "I8", say("thanks", L)
    if words(t) and set(words(t)) <= GREET_W:                   # I1 saludo puro
        return "I1", say("greet", L)
    tipo, res = search(t)                                       # I4 / I5 por tema (y tipo)
    intent = "I5" if tipo else "I4"
    if res:
        best, second = res[0], (res[1] if len(res) > 1 else None)
        st["last_id"] = best[4]["id"]
        if second is None or best[0] < second[0]:               # mejor coincidencia clara
            out = ficha(best[4], L)
            extra = [one_line(x[4]) for x in res[1:3]]
            if extra:
                out += "\n" + say("also", L) + "\n" + "\n".join(extra)
            return intent, out
        top = res[:3]
        out = say("found", L, n=len(res)) + "\n" + "\n".join(one_line(x[4]) for x in top)
        if len(res) > 3:
            out += "\n" + say("more", L, n=len(res) - 3)
        return intent, out
    if has(t, *CUES):
        return intent, say("none", L)
    if has(t, "hola", "buenas", "buenos dias", "hello", "hi", "hey", "good morning"):
        return "I1", say("greet", L)
    st["fails"] += 1
    return "I10", say("fallback", L)


In [6]:
# Celda 5: modo interactivo
def chat():
    st = new_state()
    print("Asistente:", say("greet", st["lang"]))
    while True:
        text = input("Tú: ").strip()
        if not text:
            continue
        intent, reply = respond(text, st)
        print(f"Asistente [{intent}]:", reply, "\n")
        if intent == "I9":
            break


In [7]:
# Celda 6: pruebas funcionales (12 conversaciones)
TESTS = [
 ("C1",  [("Hola", "I1")]),
 ("C2",  [("¿Qué puedes hacer?", "I2")]),
 ("C3",  [("¿Qué temas puedes consultar?", "I3")]),
 ("C4",  [("¿Qué requerimiento aplica para clips?", "I4")]),
 ("C5",  [("¿Qué regla de diseño afecta un clearance contra un sharp edge?", "I5")]),
 ("C6",  [("Dame información de DEMO-RQT-012", "I6")]),
 ("C7",  [("Dame información de DEMO-DR-007", "I6"), ("¿Qué otros están relacionados?", "I7")]),
 ("C8",  [("¿Quién es el autor de DEMO-RQT-012?", "I11")]),
 ("C9",  [("¿Cómo pido una desviación para DEMO-RQT-012?", "I12"), ("¿Y quién es el autor?", "I11")]),
 ("C10", [("Is a deviation allowed for DEMO-RQT-039?", "I12")]),
 ("C11", [("What design rules apply to grommets?", "I5"), ("thanks", "I8"), ("bye", "I9")]),
 ("C12", [("¿Qué aplica para pintura?", "I4"), ("¿Cuál es el clima hoy?", "I10")]),
]

def ejecutar_pruebas():
    ok_total = n_total = 0
    for name, turns in TESTS:
        st = new_state()
        print(f"=== {name} ===")
        for text, expected in turns:
            intent, reply = respond(text, st)
            ok = intent == expected
            ok_total += ok; n_total += 1
            print(f"Tú: {text}\nAsistente [{intent}] {'OK' if ok else 'FALLO (esperada ' + expected + ')'}:\n{reply}\n")
    print(f"Resultado: {ok_total}/{n_total} turnos con la intención esperada")

ejecutar_pruebas()
# chat()      # quita el comentario para conversar de forma interactiva


=== C1 ===
Tú: Hola
Asistente [I1] OK:
Hola. Soy el asistente de consulta de requerimientos y design rules de arneses. Los datos son sintéticos y de uso académico. Pregúntame por un tema (por ejemplo, clips) o por un ID.

=== C2 ===
Tú: ¿Qué puedes hacer?
Asistente [I2] OK:
Puedo buscar por tema (¿qué aplica para clips?), por tipo (design rules de grommets), por ID (DEMO-DR-007), mostrar registros relacionados (¿cuáles están relacionados?), el autor o la prioridad de un ID, y cómo pedir una desviación.

=== C3 ===
Tú: ¿Qué temas puedes consultar?
Asistente [I3] OK:
Temas disponibles: Retention and Clips; Protection and Edges; Routing and Clearance; Grommets and Holes; Grounding; Connectors and Sealing; Splices and Terminals; Fuses and Circuit Protection; Heat and Environment; High Voltage and Shielding

=== C4 ===
Tú: ¿Qué requerimiento aplica para clips?
Asistente [I4] OK:
Encontré 9 registros. Los más relevantes:
  - DEMO-RQT-012 (Primary): Retention forces to be withstood
  - DEMO-R